# Puxa dados dos PDF's

In [3]:
pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 503.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 106.4 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [4]:
import pdfplumber
import re
import csv
import os

In [22]:
import pdfplumber
import re
import csv
import os
import glob
import sys

# Diretórios de entrada e saída padrão do Colab
diretorio_pdfs = '/content/'
caminho_saida = '/content/DADOS_2023.csv'

# Padrão Regex para extração primária (RA, Código, Restante da Linha)
padrao_linha = re.compile(r"^(\d+)\s+([A-Za-z0-9\-]+)\s+(.+)$")

# Padrão Regex para remover a Turma, Turno, Campus e mensagens residuais do nome da disciplina
# Ele encontra o trecho " <Turma>-(Diurno|Noturno) (<Campus>)" e apaga tudo a partir dali.
padrao_limpeza_disciplina = re.compile(r'\s+[A-Za-z0-9]+-(Diurno|Noturno)\s+\(.*?\)[\s]*.*$')

dados_totais = []

# Identifica todos os arquivos PDF na pasta
arquivos_pdf = glob.glob(os.path.join(diretorio_pdfs, '*.pdf'))

if not arquivos_pdf:
    print("Nenhum arquivo PDF encontrado no diretório. Certifique-se de fazer o upload.")
else:
    for caminho_pdf in arquivos_pdf:
        nome_arquivo = os.path.basename(caminho_pdf)

        try:
            with pdfplumber.open(caminho_pdf) as pdf:
                total_paginas = len(pdf.pages)
                ultimo_percentual = -1

                for i, pagina in enumerate(pdf.pages):
                    texto = pagina.extract_text()

                    if texto:
                        for linha in texto.split('\n'):
                            match = padrao_linha.match(linha.strip())
                            if match:
                                ra = match.group(1)
                                codigo = match.group(2)
                                disciplina_suja = match.group(3).strip()

                                # Aplica a limpeza no nome da disciplina
                                disciplina_limpa = padrao_limpeza_disciplina.sub('', disciplina_suja)

                                dados_totais.append([ra, codigo, disciplina_limpa])

                    # Lógica da Barra de Progresso Visual (atualiza a cada 1% sem pular linha)
                    percentual_atual = int(((i + 1) / total_paginas) * 100)
                    if percentual_atual > ultimo_percentual:
                        ultimo_percentual = percentual_atual

                        tamanho_barra = 40
                        blocos_preenchidos = int((percentual_atual / 100) * tamanho_barra)
                        barra = '█' * blocos_preenchidos + '-' * (tamanho_barra - blocos_preenchidos)

                        sys.stdout.write(f"\rProcessando {nome_arquivo[:20].ljust(20)}: [{barra}] {percentual_atual:3d}%")
                        sys.stdout.flush()

                print() # Quebra linha ao finalizar 100% de um PDF específico

        except Exception as e:
            print(f"\nErro na leitura do arquivo {nome_arquivo}: {e}")

    # Geração do arquivo CSV unificado
    if dados_totais:
        try:
            with open(caminho_saida, mode='w', encoding='utf-8', newline='') as arquivo_csv:
                escritor = csv.writer(arquivo_csv, delimiter=';', quoting=csv.QUOTE_MINIMAL)
                escritor.writerow(['RA', 'Código_Turma', 'Nome_Disciplina'])
                escritor.writerows(dados_totais)

            print(f"\nExtração concluída com sucesso! {len(dados_totais)} matrículas unificadas em: {caminho_saida}")
        except Exception as e:
            print(f"\nErro ao salvar o arquivo final: {e}")
    else:
        print("\nNenhum dado válido extraído dos PDFs.")

Processando 20233_MATRCULAS_DEFE: [████████████████████████████████████████] 100%
Processando 2023_2_matriculas_de: [████████████████████████████████████████] 100%
Processando ajuste_2023_1_deferi: [████████████████████████████████████████] 100%

Extração concluída com sucesso! 123322 matrículas unificadas em: /content/DADOS_2023.csv


# Ingressantes

In [10]:
import pdfplumber
import re
import csv
import os
import glob
import sys
import unicodedata

# Diretórios de entrada e saída padrão do Colab
diretorio_pdfs = '/content/'
caminho_saida = '/content/INGRESSANTES_2022.csv'

# Padrão flexível para deletar turmas e campus adicionais no fim do nome
padrao_limpeza_disciplina = re.compile(
    r'\s*(?:TURMA EM INGLÊS\s*)?\s*[A-Za-z0-9]+\s*-\s*(?:diurno|noturno|matutino|vespertino|BCT|BCH)\s+\(.*$',
    flags=re.IGNORECASE
)

def formatar_disciplina(texto):
    """Limpa o final, converte para maiúsculo e remove acentos da disciplina."""
    texto_limpo = padrao_limpeza_disciplina.sub('', texto)
    texto_limpo = texto_limpo.upper()
    texto_limpo = unicodedata.normalize('NFKD', texto_limpo).encode('ASCII', 'ignore').decode('utf-8')
    return texto_limpo.strip()

dados_totais = []

# Identifica todos os arquivos PDF na pasta
arquivos_pdf = glob.glob(os.path.join(diretorio_pdfs, '*.pdf'))

if not arquivos_pdf:
    print("Nenhum arquivo PDF encontrado no diretório. Faça o upload.")
else:
    for caminho_pdf in arquivos_pdf:
        nome_arquivo = os.path.basename(caminho_pdf)

        try:
            with pdfplumber.open(caminho_pdf) as pdf:
                total_paginas = len(pdf.pages)
                ultimo_percentual = -1

                for i, pagina in enumerate(pdf.pages):
                    texto = pagina.extract_text()

                    if texto:
                        for linha in texto.split('\n'):
                            linha_limpa = linha.strip()
                            if not linha_limpa:
                                continue

                            # Tokenização: Quebra a linha inteira em blocos
                            tokens = linha_limpa.split()

                            # A linha deve começar obrigatoriamente com o número do RA
                            #if not tokens or not tokens[0].isdigit():
                            #    continue

                            ra = tokens[0][0:11]
                            codigo_turma = None
                            disciplina_start_idx = 1

                            # Avança caçando o Código da Turma
                            for idx in range(1, len(tokens)):
                                # Formato padrão UFABC: Letras + Números + Hífen + Números + Letras (Ex: NA1BHO0001-19SB)
                                if re.match(r'[A-Za-z0-9]+-\d+[A-Za-z]+', tokens[idx]):
                                    codigo_turma = tokens[idx]

                                    # Se a próxima palavra for o código curto (Ex: BHO0001-19), pula também
                                    if idx + 1 < len(tokens) and re.match(r'[A-Za-z]+\d+-\d+', tokens[idx+1]):
                                        disciplina_start_idx = idx + 2
                                    else:
                                        disciplina_start_idx = idx + 1
                                    break

                            # Se encontrou o código, limpa o resto da string e empacota para o CSV
                            if codigo_turma:
                                disciplina_suja = " ".join(tokens[disciplina_start_idx:])
                                disciplina_limpa = formatar_disciplina(disciplina_suja)

                                dados_totais.append([ra, codigo_turma, disciplina_limpa])

                    # Lógica da Barra de Progresso Visual
                    percentual_atual = int(((i + 1) / total_paginas) * 100)
                    if percentual_atual > ultimo_percentual:
                        ultimo_percentual = percentual_atual
                        tamanho_barra = 40
                        blocos_preenchidos = int((percentual_atual / 100) * tamanho_barra)
                        barra = '█' * blocos_preenchidos + '-' * (tamanho_barra - blocos_preenchidos)
                        sys.stdout.write(f"\rProcessando {nome_arquivo[:20].ljust(20)}: [{barra}] {percentual_atual:3d}%")
                        sys.stdout.flush()

                print()

        except Exception as e:
            print(f"\nErro fatal na leitura do arquivo {nome_arquivo}: {e}")

    # Exportação para CSV (Igual ao seu código original)
    if dados_totais:
        try:
            with open(caminho_saida, mode='w', encoding='utf-8', newline='') as arquivo_csv:
                escritor = csv.writer(arquivo_csv, delimiter=';', quoting=csv.QUOTE_MINIMAL)
                escritor.writerow(['RA', 'Código_Turma', 'Nome_Disciplina'])
                escritor.writerows(dados_totais)

            print(f"\nExtração concluída com sucesso! {len(dados_totais)} matrículas salvas em: {caminho_saida}")
        except Exception as e:
            print(f"\nErro ao salvar o arquivo final: {e}")
    else:
        print("\nNenhum dado válido extraído dos PDFs.")

Processando turmas_ingressantes_: [████████████████████████████████████████] 100%

Extração concluída com sucesso! 11664 matrículas salvas em: /content/INGRESSANTES_2022.csv


# Filtro de RA

In [11]:
import os
import glob
import sys
import re
import unicodedata

# Defina o RA que deseja buscar
RA_ALVO = "11202231782"
DIRETORIO_CSVS = "/content/"
ARQUIVO_SAIDA = "materiasCadastradas.txt"

def limpar_nome_disciplina(texto):
    """
    Remove informações residuais de turma, turno e campus.
    Converte para maiúsculas e remove acentuação.
    """
    # 1. Remove qualquer coisa a partir do padrão de turma.
    # Agora inclui Matutino e Vespertino, e ignora se está maiúsculo ou minúsculo.
    texto = re.sub(r'\s+[A-Za-z0-9]+-(Diurno|Noturno|Matutino|Vespertino)\s+\(.*?\)[\s]*.*$', '', texto, flags=re.IGNORECASE)

    # 2. Deixa tudo em letras maiúsculas
    texto = texto.upper()

    # 3. Remove acentuação
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')

    return texto.strip()

# Mapeia todos os CSVs no diretório
arquivos_csv = glob.glob(os.path.join(DIRETORIO_CSVS, "*.csv"))

if not arquivos_csv:
    print("Nenhum arquivo CSV encontrado no diretório.")
else:
    # Calcula o tamanho total (em bytes) de todos os CSVs para a barra de progresso
    tamanho_total_bytes = sum(os.path.getsize(arq) for arq in arquivos_csv)
    bytes_processados = 0
    ultimo_percentual = -1

    disciplinas_encontradas = []

    print(f"Iniciando varredura em {len(arquivos_csv)} arquivo(s) pelo RA {RA_ALVO}...\n")

    for caminho_arquivo in arquivos_csv:
        nome_arquivo = os.path.basename(caminho_arquivo)

        try:
            with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
                for linha in arquivo:
                    # Contabiliza o progresso baseado no tamanho em bytes da linha lida
                    bytes_processados += len(linha.encode('utf-8'))

                    # Ignora linhas vazias
                    if not linha.strip():
                        continue

                    # Divide as colunas usando o delimitador ';'
                    # Caso seus CSVs usem vírgula ',', altere o delimitador abaixo para ','
                    colunas = linha.strip().split(';')

                    # Verifica se a linha tem as colunas corretas e se o RA coincide
                    if len(colunas) >= 3 and colunas[0] == RA_ALVO:
                        disciplina_bruta = colunas[2]
                        disciplina_limpa = limpar_nome_disciplina(disciplina_bruta)
                        disciplinas_encontradas.append(disciplina_limpa)

                    # Atualização visual da barra de progresso global
                    percentual_atual = int((bytes_processados / tamanho_total_bytes) * 100)

                    if percentual_atual > ultimo_percentual:
                        ultimo_percentual = percentual_atual

                        tamanho_barra = 40
                        blocos_preenchidos = int((percentual_atual / 100) * tamanho_barra)
                        barra = '█' * blocos_preenchidos + '-' * (tamanho_barra - blocos_preenchidos)

                        sys.stdout.write(f"\rProgresso de Análise: [{barra}] {percentual_atual:3d}%")
                        sys.stdout.flush()

        except Exception as e:
            print(f"\nErro ao ler o arquivo {nome_arquivo}: {e}")

    # Remove duplicatas preservando a ordem original (Garante que nenhuma matéria se repita)
    disciplinas_unicas = list(dict.fromkeys(disciplinas_encontradas))

    # Exibição e exportação dos resultados finais
    print("\n\nResultados encontrados:")
    if disciplinas_unicas:
        print(f"Total: {len(disciplinas_unicas)} matéria(s) única(s) e limpa(s).\n")

        try:
            with open(ARQUIVO_SAIDA, mode='w', encoding='utf-8') as arquivo_txt:
                for disc in disciplinas_unicas:
                    print(disc)
                    arquivo_txt.write(disc + "\n")
            print(f"\nArquivo '{ARQUIVO_SAIDA}' gerado com sucesso!")
        except Exception as e:
            print(f"\nErro ao salvar o arquivo .txt: {e}")
    else:
        print("Nenhuma ocorrência encontrada para o RA informado em nenhum dos arquivos.")

Iniciando varredura em 1 arquivo(s) pelo RA 11202231782...

Progresso de Análise: [███████████████████████████████████████-]  98%

Resultados encontrados:
Total: 6 matéria(s) única(s) e limpa(s).

BASE EXPERIMENTAL DAS CIENCIAS NATURAIS
BASES CONCEITUAIS DA ENERGIA
ESTRUTURA DA MATERIA
EVOLUCAO E DIVERSIFICACAO DA VIDA NA TERRA
BASES MATEMATICAS
BASES COMPUTACIONAIS DA CIENCIA

Arquivo 'materiasCadastradas.txt' gerado com sucesso!
